In [1]:
!pip install playwright python-dotenv pandas
!python -m playwright install

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/37.9 MB ? eta -:--:--
    --------------------------------------- 0.8/37.9 MB 2.4 MB/s eta 0:00:16
   -- ------------------------------------- 2.1/37.9 MB 3.8 MB/s eta 0:00:10
   ---- ----------------------------------- 4.2/37.9 MB 6.0 MB/s eta 0:00:06
   ------ --------------------------------- 6.0/37.9 MB 7.4 MB/s eta 0:00:05
   --------- ------------------------------ 9.2/37.9 MB 8.0 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ------------ -------

In [2]:
env_content = """
PORTAL_USERNAME=2380223
PORTAL_PASSWORD=Muneeb@Anjum3543
"""

with open(".env", "w", encoding="utf-8") as file:
    file.write(env_content.strip())

print(".env file created successfully.")

.env file created successfully.


In [3]:
gitignore_content = """
.env
__pycache__/
*.pkl
*.html
*.png
"""

with open(".gitignore", "w", encoding="utf-8") as file:
    file.write(gitignore_content.strip())

print(".gitignore file created successfully.")

.gitignore file created successfully.


In [4]:
scraper_code = r'''
import time
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"

def main():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page()

        page.goto(PORTAL_URL)
        print("Portal opened.")

        print("Login manually in the browser.")
        print("After login, go to Attendance / Marks page.")
        print("You have 120 seconds.")

        time.sleep(120)

        page.screenshot(path="portal_after_login.png", full_page=True)

        html = page.content()
        with open("portal_after_login.html", "w", encoding="utf-8") as file:
            file.write(html)

        print("Saved portal_after_login.png")
        print("Saved portal_after_login.html")

        browser.close()

if __name__ == "__main__":
    main()
'''

with open("scraper.py", "w", encoding="utf-8") as file:
    file.write(scraper_code)

print("scraper.py created successfully.")

scraper.py created successfully.


In [5]:
!python scraper.py

Portal opened.
Login manually in the browser.
After login, go to Attendance / Marks page.
You have 120 seconds.
Saved portal_after_login.png
Saved portal_after_login.html


In [12]:
scraper_code = r'''
import time
import re
from pathlib import Path
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

OUTPUT_DIR = Path("portal_captures")
OUTPUT_DIR.mkdir(exist_ok=True)


def save_page(page, filename_prefix):
    html_path = OUTPUT_DIR / f"{filename_prefix}.html"
    png_path = OUTPUT_DIR / f"{filename_prefix}.png"

    with open(html_path, "w", encoding="utf-8") as file:
        file.write(page.content())

    page.screenshot(path=str(png_path), full_page=True)

    print(f"Saved {html_path}")
    print(f"Saved {png_path}")


def clean_filename(text):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def get_link_href(page, text_value):
    locator = page.locator(f"a:has-text('{text_value}')").first
    href = locator.get_attribute("href")

    if not href:
        raise ValueError(f"Could not find href for: {text_value}")

    if href.startswith("/"):
        href = BASE_URL + href

    return href


def get_course_links(page):
    courses = []

    links = page.locator("a").all()

    for link in links:
        text = link.inner_text().strip()
        href = link.get_attribute("href")

        if href and "chkSubmit" in href:
            courses.append({
                "course_name": text,
                "href": href
            })

    return courses


def capture_course_details(page, main_url, page_type):
    print(f"\nOpening {page_type} main page...")
    page.goto(main_url)
    time.sleep(5)

    save_page(page, f"{page_type}_main_page")

    courses = get_course_links(page)
    print(f"Found {len(courses)} courses on {page_type} page.")

    for index, course in enumerate(courses, start=1):
        course_name = course["course_name"]
        safe_name = clean_filename(course_name)

        print(f"\n{page_type.upper()} {index}: {course_name}")

        page.goto(main_url)
        time.sleep(2)

        course_locator = page.locator(f"a:has-text('{course_name}')").first
        course_locator.click()

        time.sleep(5)

        save_page(page, f"{page_type}_{index}_{safe_name}")


def main():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1400, "height": 900})

        page.goto(PORTAL_URL)
        print("Portal opened.")

        print("\nLOGIN MANUALLY.")
        print("After login, wait. Do not close the browser.")
        print("You have 60 seconds.\n")
        time.sleep(60)

        save_page(page, "logged_in_homepage")

        attendance_url = get_link_href(page, "View Attendance")
        results_url = get_link_href(page, "Current Semester Results")

        print("\nAttendance URL:", attendance_url)
        print("Results URL:", results_url)

        capture_course_details(page, attendance_url, "attendance")
        capture_course_details(page, results_url, "results")

        print("\nDone. All detail pages saved inside portal_captures folder.")
        browser.close()


if __name__ == "__main__":
    main()
'''

with open("scraper.py", "w", encoding="utf-8") as file:
    file.write(scraper_code)

print("scraper.py updated successfully.")

scraper.py updated successfully.


In [13]:
!python scraper.py

Portal opened.

LOGIN MANUALLY.
After login, wait. Do not close the browser.
You have 60 seconds.

Saved portal_captures\logged_in_homepage.html
Saved portal_captures\logged_in_homepage.png

Attendance URL: https://springzabdesk.szabist-isb.edu.pk/Student/QryCourseAttendance.asp?OptionName=View Attendance&sid=577339771
Results URL: https://springzabdesk.szabist-isb.edu.pk/Student/QryCourseRecapSheet.asp?OptionName=Current Semester Results&sid=577339771

Opening attendance main page...
Saved portal_captures\attendance_main_page.html
Saved portal_captures\attendance_main_page.png
Found 8 courses on attendance page.

ATTENDANCE 1: CSC 4102 Professional Practices
Saved portal_captures\attendance_1_csc_4102_professional_practices.html
Saved portal_captures\attendance_1_csc_4102_professional_practices.png

ATTENDANCE 2: SEC 4516 Artificial Intelligence
Saved portal_captures\attendance_2_sec_4516_artificial_intelligence.html
Saved portal_captures\attendance_2_sec_4516_artificial_intelligence.

In [14]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(text.replace("\xa0", " ").split())

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    rows = soup.find_all("tr")

    for row in rows:
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_attendance_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    total_lectures = 0
    present = 0
    absent = 0
    late = 0

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all("td")]

        if len(cells) == 3 and cells[0].isdigit():
            total_lectures += 1
            status = cells[2].lower()

            if status == "present":
                present += 1
            elif status == "absent":
                absent += 1
            elif status == "late":
                late += 1

    attendance_percentage = 0

    if total_lectures > 0:
        attendance_percentage = round(((present + late) / total_lectures) * 100, 2)

    return {
        "subject": course,
        "instructor": instructor,
        "program": program,
        "section": section,
        "total_lectures": total_lectures,
        "total_present": present,
        "total_absent": absent,
        "total_late": late,
        "attendance_percentage": attendance_percentage
    }

def main():
    attendance_files = sorted(CAPTURE_DIR.glob("attendance_[0-9]*_*.html"))

    if not attendance_files:
        print("No attendance detail files found.")
        return

    rows = []

    for file_path in attendance_files:
        rows.append(parse_attendance_file(file_path))

    df = pd.DataFrame(rows)

    df["attendance_risk"] = df["attendance_percentage"].apply(
        lambda x: "High Risk" if x < 75 else "Warning" if x < 80 else "Safe"
    )

    df.to_csv("scraped_attendance_summary.csv", index=False)

    print("Saved scraped_attendance_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_attendance.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_attendance.py created successfully.")

parse_attendance.py created successfully.


In [15]:
!python parse_attendance.py

Saved scraped_attendance_summary.csv
                                      subject  ... attendance_risk
0                      Professional Practices  ...            Safe
1                     Artificial Intelligence  ...       High Risk
2      Formal Methods in Software Engineering  ...       High Risk
3                             Web Engineering  ...            Safe
4                        Information Security  ...            Safe
5                     Teachings of Holy Quran  ...       High Risk
6       Software Construction and Development  ...            Safe
7  Lab: Software Construction and Development  ...         Warning

[8 rows x 10 columns]


In [16]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(text.replace("\xa0", " ").split())

def to_number(value):
    value = clean(str(value))

    if value.lower() == "not entered":
        return 0

    try:
        return float(value)
    except:
        return 0

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_result_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    quiz_marks = 0
    assignment_marks = 0
    mid_marks = 0
    final_marks = 0
    total_marks = 0
    total_percentage = 0
    grade = "-"
    reason = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) == 3:
            head = cells[0]
            obtained = cells[2]

            if head.startswith("Quiz ("):
                quiz_marks = to_number(obtained)

            elif head.startswith("Assignment ("):
                assignment_marks = to_number(obtained)

            elif head.startswith("Mid Term Paper ("):
                mid_marks = to_number(obtained)

            elif head.startswith("Final Paper ("):
                final_marks = to_number(obtained)

            elif head == "Total Marks":
                match = re.search(r"([\d.]+)\s*/\s*100\s*\((\d+)%\)", obtained)
                if match:
                    total_marks = float(match.group(1))
                    total_percentage = float(match.group(2))

            elif head == "Grade":
                grade = obtained

            elif head == "Reason":
                reason = obtained

    return {
        "subject": course,
        "program": program,
        "section": section,
        "instructor": instructor,
        "quiz_marks": quiz_marks,
        "assignment_marks": assignment_marks,
        "mid_marks": mid_marks,
        "final_marks": final_marks,
        "total_obtained_marks": total_marks,
        "current_marks_percentage": total_percentage,
        "grade": grade,
        "reason": reason
    }

def main():
    result_files = sorted(CAPTURE_DIR.glob("results_[0-9]*_*.html"))

    if not result_files:
        print("No result detail files found.")
        return

    rows = []

    for file_path in result_files:
        rows.append(parse_result_file(file_path))

    df = pd.DataFrame(rows)

    df["marks_risk"] = df["current_marks_percentage"].apply(
        lambda x: "High Risk" if x < 50 else "Warning" if x < 65 else "Safe"
    )

    df.to_csv("scraped_marks_summary.csv", index=False)

    print("Saved scraped_marks_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_marks.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_marks.py created successfully.")

parse_marks.py created successfully.


In [17]:
!python parse_marks.py

Saved scraped_marks_summary.csv
                                      subject program  ... reason marks_risk
0                      Professional Practices  BSSE-6  ...         High Risk
1                     Artificial Intelligence  BSSE-6  ...         High Risk
2      Formal Methods in Software Engineering  BSSE-6  ...         High Risk
3                             Web Engineering  BSSE-6  ...         High Risk
4                        Information Security  BSSE-6  ...         High Risk
5                     Teachings of Holy Quran  BSSE-6  ...         High Risk
6       Software Construction and Development  BSSE-5  ...         High Risk
7  Lab: Software Construction and Development  BSSE-5  ...         High Risk

[8 rows x 13 columns]


In [18]:
merge_code = r'''
import pandas as pd

attendance = pd.read_csv("scraped_attendance_summary.csv")
marks = pd.read_csv("scraped_marks_summary.csv")

final_df = pd.merge(
    attendance,
    marks,
    on=["subject", "program", "section", "instructor"],
    how="outer"
)

def final_risk(row):
    attendance = row.get("attendance_percentage", 0)
    marks = row.get("current_marks_percentage", 0)
    reason = str(row.get("reason", "")).lower()

    if attendance < 75 or marks < 50 or "short attendance" in reason:
        return "High Risk"

    if attendance < 80 or marks < 65:
        return "Warning"

    return "Safe"

def recommendation(row):
    recs = []

    if row["attendance_percentage"] < 75:
        recs.append("Attendance is below 75%. Attend every upcoming class.")

    elif row["attendance_percentage"] < 80:
        recs.append("Attendance is close to danger zone. Avoid absents.")

    if row["current_marks_percentage"] < 50:
        recs.append("Marks are weak. Focus strongly before finals.")

    elif row["current_marks_percentage"] < 65:
        recs.append("Marks are average. Improve quizzes, assignments, and final prep.")

    if "short attendance" in str(row.get("reason", "")).lower():
        recs.append("Portal shows short attendance issue. Confirm with exam/department office.")

    if row["final_marks"] == 0:
        recs.append("Final marks are not entered yet, so current result is incomplete.")

    if not recs:
        recs.append("Subject looks stable. Maintain performance.")

    return " ".join(recs)

final_df["final_risk_status"] = final_df.apply(final_risk, axis=1)
final_df["recommendation"] = final_df.apply(recommendation, axis=1)

final_df.to_csv("final_scraped_academic_dashboard.csv", index=False)

print("Saved final_scraped_academic_dashboard.csv")
print(final_df)
'''

with open("merge_dashboard.py", "w", encoding="utf-8") as file:
    file.write(merge_code)

print("merge_dashboard.py created successfully.")

merge_dashboard.py created successfully.


In [19]:
!python merge_dashboard.py

Saved final_scraped_academic_dashboard.csv
                                      subject  ...                                     recommendation
0                     Artificial Intelligence  ...  Attendance is below 75%. Attend every upcoming...
1      Formal Methods in Software Engineering  ...  Attendance is below 75%. Attend every upcoming...
2                        Information Security  ...  Marks are weak. Focus strongly before finals. ...
3  Lab: Software Construction and Development  ...  Attendance is close to danger zone. Avoid abse...
4                      Professional Practices  ...  Marks are weak. Focus strongly before finals. ...
5       Software Construction and Development  ...  Marks are weak. Focus strongly before finals. ...
6                     Teachings of Holy Quran  ...  Attendance is below 75%. Attend every upcoming...
7                             Web Engineering  ...  Marks are weak. Focus strongly before finals. ...

[8 rows x 21 columns]


In [20]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(str(text).replace("\xa0", " ").split())

def to_number(value):
    value = clean(value)

    if value.lower() in ["not entered", "-", ""]:
        return 0

    try:
        return float(value)
    except:
        return 0

def extract_total_marks(value):
    value = clean(value)
    match = re.search(r"([\d.]+)\s*/\s*100\s*\((\d+)%\)", value)

    if match:
        return float(match.group(1)), float(match.group(2))

    return 0, 0

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_result_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    quiz_marks = 0
    assignment_marks = 0
    mid_marks = 0
    final_marks = 0
    total_marks = 0
    total_percentage = 0
    grade = "-"
    reason = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) < 2:
            continue

        head = cells[0]
        obtained = cells[-1]

        if head.startswith("Quiz ("):
            quiz_marks = to_number(obtained)

        elif head.startswith("Assignment ("):
            assignment_marks = to_number(obtained)

        elif head.startswith("Mid Term Paper ("):
            mid_marks = to_number(obtained)

        elif head.startswith("Final Paper ("):
            final_marks = to_number(obtained)

        elif head == "Total Marks":
            total_marks, total_percentage = extract_total_marks(obtained)

        elif head == "Grade":
            grade = obtained

        elif head == "Reason":
            reason = obtained

    return {
        "subject": course,
        "program": program,
        "section": section,
        "instructor": instructor,
        "quiz_marks": quiz_marks,
        "assignment_marks": assignment_marks,
        "mid_marks": mid_marks,
        "final_marks": final_marks,
        "total_obtained_marks": total_marks,
        "current_marks_percentage": total_percentage,
        "grade": grade,
        "reason": reason
    }

def main():
    result_files = sorted(CAPTURE_DIR.glob("results_[0-9]*_*.html"))

    if not result_files:
        print("No result detail files found.")
        return

    rows = []

    for file_path in result_files:
        rows.append(parse_result_file(file_path))

    df = pd.DataFrame(rows)

    df["marks_risk"] = df["current_marks_percentage"].apply(
        lambda x: "High Risk" if x < 50 else "Warning" if x < 65 else "Safe"
    )

    df.to_csv("scraped_marks_summary.csv", index=False)

    print("Saved scraped_marks_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_marks.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_marks.py fixed successfully.")

parse_marks.py fixed successfully.


In [21]:
!python parse_marks.py

Saved scraped_marks_summary.csv
                                      subject  ... marks_risk
0                      Professional Practices  ...  High Risk
1                     Artificial Intelligence  ...  High Risk
2      Formal Methods in Software Engineering  ...  High Risk
3                             Web Engineering  ...  High Risk
4                        Information Security  ...  High Risk
5                     Teachings of Holy Quran  ...  High Risk
6       Software Construction and Development  ...  High Risk
7  Lab: Software Construction and Development  ...    Warning

[8 rows x 13 columns]


In [22]:
!python merge_dashboard.py

Saved final_scraped_academic_dashboard.csv
                                      subject  ...                                     recommendation
0                     Artificial Intelligence  ...  Attendance is below 75%. Attend every upcoming...
1      Formal Methods in Software Engineering  ...  Attendance is below 75%. Attend every upcoming...
2                        Information Security  ...  Marks are weak. Focus strongly before finals. ...
3  Lab: Software Construction and Development  ...  Attendance is close to danger zone. Avoid abse...
4                      Professional Practices  ...  Marks are weak. Focus strongly before finals. ...
5       Software Construction and Development  ...  Marks are weak. Focus strongly before finals. ...
6                     Teachings of Holy Quran  ...  Attendance is below 75%. Attend every upcoming...
7                             Web Engineering  ...  Marks are weak. Focus strongly before finals. ...

[8 rows x 21 columns]


In [24]:
from pathlib import Path
import shutil

ROOT = Path.cwd()

folders = [
    "data/raw",
    "data/processed",
    "data/summaries",
    "scripts",
    "assets/screenshots",
]

for folder in folders:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

move_map = {
    "academic_data.csv": "data/raw/academic_data.csv",
    "final_academic_dashboard.csv": "data/processed/final_academic_dashboard.csv",
    "final_scraped_academic_dashboard.csv": "data/processed/final_scraped_academic_dashboard.csv",
    "scraped_attendance_summary.csv": "data/summaries/scraped_attendance_summary.csv",
    "scraped_marks_summary.csv": "data/summaries/scraped_marks_summary.csv",
    "scraper.py": "scripts/scraper.py",
    "parse_attendance.py": "scripts/parse_attendance.py",
    "parse_marks.py": "scripts/parse_marks.py",
    "merge_dashboard.py": "scripts/merge_dashboard.py",
    "portal_after_login.html": "data/raw/portal_after_login.html",
    "portal_after_login.htm": "data/raw/portal_after_login.htm",
    "portal_after_login.png": "assets/screenshots/portal_after_login.png",
}

for source, destination in move_map.items():
    source_path = ROOT / source
    destination_path = ROOT / destination

    if source_path.exists():
        if destination_path.exists():
            destination_path.unlink()
        shutil.move(str(source_path), str(destination_path))
        print(f"Moved: {source} -> {destination}")
    else:
        print(f"Skipped missing: {source}")

portal_captures_source = ROOT / "portal_captures"
portal_captures_destination = ROOT / "data/raw/portal_captures"

if portal_captures_source.exists():
    if portal_captures_destination.exists():
        shutil.rmtree(portal_captures_destination)
    shutil.move(str(portal_captures_source), str(portal_captures_destination))
    print("Moved: portal_captures -> data/raw/portal_captures")
else:
    print("Skipped missing: portal_captures")

gitignore_content = "\n".join([
    ".env",
    "__pycache__/",
    ".ipynb_checkpoints/",
    "*.pkl",
    "data/raw/portal_after_login.*",
    "data/raw/portal_captures/",
    "assets/screenshots/",
])

(ROOT / ".gitignore").write_text(gitignore_content, encoding="utf-8")

readme_lines = [
    "# SZABIST Academic Performance Dashboard",
    "",
    "## Project Overview",
    "",
    "This project analyzes academic performance using attendance and marks data from the SZABIST ZABDESK student portal.",
    "",
    "The dashboard tracks subject-wise attendance, present/absent counts, quiz marks, assignment marks, midterm marks, final marks, total marks, grade status, and academic risk.",
    "",
    "## Main Features",
    "",
    "- Scrapes academic portal pages using Playwright",
    "- Extracts subject-wise attendance data",
    "- Extracts quiz, assignment, midterm, final, total marks, grade, and result reason",
    "- Calculates attendance percentage",
    "- Detects academic risk",
    "- Generates CSV summaries",
    "- Builds a Jupyter Notebook dashboard",
    "- Keeps private scraped HTML files out of GitHub",
    "",
    "## Project Structure",
    "",
    "szabist-academic-dashboard/",
    "├── README.md",
    "├── .gitignore",
    "├── SZABIST_Academic_Dashboard.ipynb",
    "├── scraper.ipynb",
    "├── data/",
    "│   ├── raw/",
    "│   ├── processed/",
    "│   └── summaries/",
    "├── scripts/",
    "└── assets/",
    "",
    "## Technologies Used",
    "",
    "- Python",
    "- Jupyter Notebook",
    "- Pandas",
    "- BeautifulSoup",
    "- Playwright",
    "- Matplotlib",
    "",
    "## Important Files",
    "",
    "| File | Purpose |",
    "|---|---|",
    "| SZABIST_Academic_Dashboard.ipynb | Main analysis and dashboard notebook |",
    "| scraper.ipynb | Setup, scraping, parsing, and file management notebook |",
    "| scripts/scraper.py | Opens portal and captures attendance/result pages |",
    "| scripts/parse_attendance.py | Extracts attendance summary |",
    "| scripts/parse_marks.py | Extracts marks summary |",
    "| scripts/merge_dashboard.py | Merges attendance and marks into final dashboard CSV |",
    "| data/processed/final_scraped_academic_dashboard.csv | Final real scraped dashboard data |",
    "",
    "## Final Dataset",
    "",
    "Main output file:",
    "",
    "data/processed/final_scraped_academic_dashboard.csv",
    "",
    "## Privacy Warning",
    "",
    "Do not upload these files/folders to GitHub:",
    "",
    "- .env",
    "- data/raw/portal_after_login.*",
    "- data/raw/portal_captures/",
    "- assets/screenshots/",
    "",
    "These may contain private academic or personal portal data.",
    "",
    "## How to Use",
    "",
    "Open SZABIST_Academic_Dashboard.ipynb and load:",
    "",
    "import pandas as pd",
    "df = pd.read_csv('data/processed/final_scraped_academic_dashboard.csv')",
    "df",
]

(ROOT / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")

print("Project folder sorted successfully.")
print("README.md updated.")
print(".gitignore updated.")

Moved: academic_data.csv -> data/raw/academic_data.csv
Moved: final_academic_dashboard.csv -> data/processed/final_academic_dashboard.csv
Moved: final_scraped_academic_dashboard.csv -> data/processed/final_scraped_academic_dashboard.csv
Moved: scraped_attendance_summary.csv -> data/summaries/scraped_attendance_summary.csv
Moved: scraped_marks_summary.csv -> data/summaries/scraped_marks_summary.csv
Moved: scraper.py -> scripts/scraper.py
Moved: parse_attendance.py -> scripts/parse_attendance.py
Moved: parse_marks.py -> scripts/parse_marks.py
Moved: merge_dashboard.py -> scripts/merge_dashboard.py
Moved: portal_after_login.html -> data/raw/portal_after_login.html
Skipped missing: portal_after_login.htm
Moved: portal_after_login.png -> assets/screenshots/portal_after_login.png
Moved: portal_captures -> data/raw/portal_captures
Project folder sorted successfully.
README.md updated.
.gitignore updated.
